# 00_core — Create canonical splits and TF-IDF vectorizer

Owner: Keana Gindlesperger

Reads: stanfordnlp/imdb (Hugging Face)
Writes: data/processed/splits.parquet, artifacts/tfidf_vectorizer.joblib

Notes: This notebook creates the canonical `splits.parquet` used by the pipeline
and fits the TF-IDF vectorizer on the `fit` split only. The notebook is written
to be explicit about seeds and artifact locations so downstream notebooks can
rely on the immutable disk-handoff contract.


In [1]:
# Bootstrap import of src/shared.py (follows repo guidance)
import pathlib
import sys

import pandas as pd
from sklearn.model_selection import train_test_split

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
from shared import (
    PATHS,
    PREDICTION_SCHEMA,
    SEED,
    SPLIT_FINGERPRINTS,
    SPLIT_SIZES,
    fingerprint_texts,
    fit_and_save_vectorizer,
)

print("SEED =", SEED)
# Print paths relative to the repo root: this is a public repo, so no local
# filesystem layout belongs in a committed output cell.
ROOT = PATHS["repo_root"]
print("splits target =", PATHS["splits_parquet"].relative_to(ROOT))
print("tfidf target =", PATHS["tfidf"].relative_to(ROOT))
print("split sizes =", SPLIT_SIZES)
print("prediction schema =", PREDICTION_SCHEMA)


SEED = 42
splits target = data/processed/splits.parquet
tfidf target = artifacts/tfidf_vectorizer.joblib
split sizes = {'fit': 10000, 'val': 5000, 'test': 25000}
prediction schema = ['id', 'y_true', 'y_pred', 'y_proba_pos']


## Dependencies
The code below uses the Hugging Face `datasets` library to download `stanfordnlp/imdb`
and requires a parquet engine such as `pyarrow` to write parquet files. If these
are not installed in the environment, run `uv add datasets pyarrow` and then `uv sync`
before re-running this notebook.


In [2]:
# Attempt to import datasets and provide a friendly error if missing
try:
    from datasets import load_dataset
except Exception as e:
    raise RuntimeError(
        "Missing dependency: `datasets`. Install with `uv add datasets` then `uv sync`."
    ) from e

# Load the HF dataset (may take a while the first time)
print('Downloading stanfordnlp/imdb from Hugging Face...')
ds = load_dataset('stanfordnlp/imdb')
print('Dataset splits:', ds.keys())


Dataset splits: dict_keys(['train', 'test', 'unsupervised'])


In [3]:
# Use only labeled Hugging Face splits.
# fit and val are sampled from ds["train"]; ds["test"] remains untouched.
pool = pd.DataFrame(ds["train"])[["text", "label"]].copy()
test_df = pd.DataFrame(ds["test"])[["text", "label"]].copy()

# Keep each review's origin. The DataFrame index is still the Hugging Face row
# number here, so capture it before any sampling or reindexing discards it.
pool["source_split"], pool["source_index"] = "train", pool.index
test_df["source_split"], test_df["source_index"] = "test", test_df.index

print("HF train rows:", len(pool))
print("HF test rows:", len(test_df))
print("HF train labels:", sorted(pool["label"].unique()))
print("HF test labels:", sorted(test_df["label"].unique()))


HF train rows: 25000
HF test rows: 25000
HF train labels: [np.int64(0), np.int64(1)]
HF test labels: [np.int64(0), np.int64(1)]


In [4]:
# Create canonical, stratified fit/validation splits from HF train only.
fit_df, val_df = train_test_split(
    pool,
    train_size=SPLIT_SIZES["fit"],
    test_size=SPLIT_SIZES["val"],
    stratify=pool["label"],
    random_state=SEED,
)

fit_df = fit_df.reset_index(drop=True).copy()
val_df = val_df.reset_index(drop=True).copy()
test_df = test_df.reset_index(drop=True).copy()

fit_df["split"] = "fit"
val_df["split"] = "val"
test_df["split"] = "test"

# Stable IDs are unique across splits and reproducible for the fixed seed.
fit_df["id"] = [f"fit-{i:06d}" for i in range(len(fit_df))]
val_df["id"] = [f"val-{i:06d}" for i in range(len(val_df))]
test_df["id"] = [f"test-{i:06d}" for i in range(len(test_df))]

assert "id" in PREDICTION_SCHEMA, "Prediction schema must include id"
df_out = pd.concat([fit_df, val_df, test_df], ignore_index=True)[
    ["id", "text", "label", "split", "source_split", "source_index"]
]

# Verify before writing any artifacts.
expected_sizes = {
    "fit": SPLIT_SIZES["fit"],
    "val": SPLIT_SIZES["val"],
    "test": SPLIT_SIZES["test"],
}
actual_sizes = df_out.groupby("split").size().to_dict()
assert actual_sizes == expected_sizes, (
    f"Unexpected split sizes: {actual_sizes}; expected {expected_sizes}"
)
assert set(df_out["label"]) == {0, 1}, (
    f"Expected binary labels {{0, 1}}, found {sorted(df_out['label'].unique())}"
)
assert df_out["id"].is_unique, "IDs must be unique across all splits"

# Fingerprint each split and compare against the ratified values. Ids are
# positional, so a divergent sample on another machine would still produce
# ids that look identical while pointing at different reviews. This turns that
# silent failure into a stop.
fingerprints = {
    name: fingerprint_texts(df_out.loc[df_out["split"] == name, "text"])
    for name in ("fit", "val", "test")
}
print("Split fingerprints:", fingerprints)
assert fingerprints == SPLIT_FINGERPRINTS, (
    f"Split fingerprints do not match the ratified values.\n"
    f"  got:      {fingerprints}\n"
    f"  expected: {SPLIT_FINGERPRINTS}\n"
    "This machine produced a different sample. Check that the environment came "
    "from `uv sync` (a different scikit-learn can reshuffle train_test_split) "
    "before using any artifact from this run."
)
assert df_out["source_index"].notna().all(), "Provenance columns must be populated"

class_counts = df_out.groupby(["split", "label"]).size()
print("Using prediction schema:", PREDICTION_SCHEMA)
print("Per-split class counts:")
print(class_counts)

# Write the validated canonical parquet artifact.
p = PATHS["splits_parquet"]
p.parent.mkdir(parents=True, exist_ok=True)
try:
    df_out.to_parquet(p, index=False)
    print("Wrote splits.parquet ->", p.relative_to(PATHS["repo_root"]))
except Exception as e:
    raise RuntimeError(
        "Failed to write parquet. Ensure pyarrow is installed (uv add pyarrow)."
    ) from e


Split fingerprints: {'fit': 'a119c174af55823c', 'val': 'f10ba2466b84170b', 'test': '56278a6aa6fbfb16'}
Using prediction schema: ['id', 'y_true', 'y_pred', 'y_proba_pos']
Per-split class counts:
split  label
fit    0         5000
       1         5000
test   0        12500
       1        12500
val    0         2500
       1         2500
dtype: int64
Wrote splits.parquet -> data/processed/splits.parquet


In [5]:
# Fit TF-IDF on the 'fit' split and save the artifact intentionally
texts_fit = df_out.loc[df_out['split'] == 'fit', 'text'].astype(str).tolist()
print('Fitting TF-IDF on', len(texts_fit), 'documents (fit split)')
# This will write PATHS['tfidf']. Pass overwrite=True here for the initial run.
vec = fit_and_save_vectorizer(texts_fit, overwrite=True)
print('TF-IDF fitted and saved to', PATHS['tfidf'].relative_to(PATHS['repo_root']))


Fitting TF-IDF on 10000 documents (fit split)


TF-IDF fitted and saved to artifacts/tfidf_vectorizer.joblib


## Summary
The canonical splits parquet and TF-IDF artifact have been created. Fit and validation
come only from the labeled Hugging Face training split, while the full 25,000-row
Hugging Face test split is preserved. Binary-label, split-size, and split-fingerprint assertions run before
artifacts are written, and each row keeps its Hugging Face origin in
`source_split` / `source_index`. Downstream notebooks should import `PATHS` and `load_features`
from `shared` and use the saved artifacts.


In [6]:
print('Artifacts:')
print('  splits ->', PATHS['splits_parquet'].relative_to(PATHS['repo_root']))
print('  tfidf  ->', PATHS['tfidf'].relative_to(PATHS['repo_root']))


Artifacts:
  splits -> data/processed/splits.parquet
  tfidf  -> artifacts/tfidf_vectorizer.joblib
